In [ ]:
#| default_exp nc2csv

# NetCDF to CSV

> Convert MARIS standard NetCDF4 files into CSV files for the legacy SQL import pipeline.

The NetCDF file is the canonical curated format. This module is an output serializer, it reads a standardised NetCDF, decodes enumerated values back to display names, adds sample type and taxon information, and writes one CSV per sample group with column names the MARIS master DB SQL import script expects.

## Dependencies

In [ ]:
#| export
from pathlib import Path
from netCDF4 import Dataset
import pandas as pd
from marisco.configs import CSV_VARS, CSV_DTYPES, get_lut, get_time_units, lut_fname, lut_path, SMP_TYPE_LUT, NC_VARS
from marisco.utils import ExtractNetcdfContents
from cftime import num2date

In [ ]:
#| hide
from fastcore.test import test_eq
import pandas as pd

## Helpers

Each step is a single-responsibility function that transforms a dict of DataFrames keyed by sample group. They are composed in `to_csv()` below.

### Read NetCDF groups

Open a standardised MARIS NetCDF file, extract each group's variable arrays, and return them as `{group_name: DataFrame}`.


In [ ]:
#| export
def read_nc_grps(
    fname # Path to netcdf file
    ):
    "Read a MARIS NetCDF file and return {group: DataFrame} dict."
    with Dataset(fname, 'r') as nc:
        dfs = {}
        for gn, grp in nc.groups.items():
            data = {vn: grp.variables[vn][:] for vn in grp.variables if vn not in grp.dimensions}
            df = pd.DataFrame(data)
            rename = {n: k for k, n in NC_VARS.items() if n in df.columns}
            dfs[gn.upper()] = df.rename(columns=rename)
    return dfs

In [ ]:
#| eval: false
fname_in = Path('files/nc/100_HELCOM_MORS_2024.nc')
dfs = read_nc_grps(fname_in)
print('groups: ', list(dfs.keys()), '\n')
print('A single row: ', dfs['BIOTA'].sample(1).T)

groups:  ['BIOTA', 'SEAWATER', 'SEDIMENT'] 


A single row:                              3
SMP_ID_PROVIDER  BCLOR1992003
LON                      19.0
LAT                 54.583302
SMP_DEPTH                 0.0
TIME                701740800
STATION                GD.BAY
NUCLIDE                    31
VALUE                     1.9
UNIT                        5
UNC                     0.532
DL                          1
BIO_GROUP                   4
SPECIES                   192
BODY_PART                  52
DRYWT               75.900002
WETWT                   230.0
PERCENTWT                0.33


### Filter columns

Keep only columns in the CSV schema. The NetCDF has internal columns (like `SMP_ID`) not part of the CSV output.

In [ ]:
#| export
def keep_csv_cols(
    dfs:dict,           # dict of DataFrames keyed by sample group
    cols:list=CSV_VARS  # columns to keep (defaults to CSV_VARS)
):
    "Keep only columns listed in `cols`."
    return {g: df[[c for c in cols if c in df.columns]] for g, df in dfs.items()}

In [ ]:
test = {'SEAWATER': pd.DataFrame({'SMP_ID': [1], 'TIME': [1], 'VALUE': [1], 'EXTRA': [1]})}
res = keep_csv_cols(test)
test_eq(list(res['SEAWATER'].columns), ['TIME', 'VALUE'])

### Decode time

NetCDF stores time as seconds since epoch. Convert to datetime.

In [ ]:
#| export
def decode_time(
    dfs:dict          # dict of DataFrames keyed by sample group
):
    "Decode TIME from epoch seconds to datetime."
    units = get_time_units()
    for df in dfs.values():
        df['TIME'] = df['TIME'].apply(lambda x: num2date(x, units=units, only_use_cftime_datetimes=False))

In [ ]:
test = {'SEAWATER': pd.DataFrame({'TIME': [1672531200]})}
decode_time(test)
test_eq(test['SEAWATER']['TIME'].iloc[0], pd.Timestamp('2023-01-01'))

### Add sample type

SQL import expects a sample type column. Each group has a fixed identifier.

In [ ]:
#| export
def add_sample_type(
    dfs:dict      # dict of DataFrames keyed by sample group
):
    "Add SAMPLE_TYPE column using group ID mapping."
    for grp, df in dfs.items(): df['SAMPLE_TYPE'] = SMP_TYPE_LUT[grp]

In [ ]:
test = {'SEAWATER': pd.DataFrame({'VALUE': [1]}), 'BIOTA': pd.DataFrame({'VALUE': [2]})}
add_sample_type(test)
test_eq(test['SEAWATER']['SAMPLE_TYPE'].iloc[0], 1)
test_eq(test['BIOTA']['SAMPLE_TYPE'].iloc[0], 2)

### Add reference ID

Optional column from Zotero / INIS archive location. Omitted if not provided.

In [ ]:
#| export
def add_ref_id(
    dfs:dict,          # dict of DataFrames keyed by sample group
    ref_id:int=None         # Reference ID to add as REF_ID column
):
    "Add REF_ID column if `ref_id` is provided."
    if ref_id is None: return
    for df in dfs.values(): df['REF_ID'] = ref_id

In [ ]:
test = {'SEAWATER': pd.DataFrame({'VALUE': [1]})}
add_ref_id(test)
test_eq('REF_ID' in test['SEAWATER'].columns, False)

In [ ]:
test = {'SEAWATER': pd.DataFrame({'VALUE': [1]})}
add_ref_id(test, ref_id=42)
test_eq(test['SEAWATER']['REF_ID'].iloc[0], 42)

### Add taxon information

Map BIOTA species IDs to scientific names and database references via the MARIS species lookup table.

In [ ]:
#| exports
TAXON_COLS = {
    'Taxonname': 'TAXONNAME',
    'Taxonrank': 'TAXONRANK',
    'TaxonDB': 'TAXONDB',
    'TaxonDBID': 'TAXONDBID',
    'TaxonDBURL': 'TAXONDBURL',
}

In [ ]:
#| export
def get_taxon_cols()->dict:       # {col_name: {species_id: value}} mapping
    "Read species lookup table, return `{col_name: {species_id: value}}` dict."
    f = Path(lut_path()) / lut_fname('SPECIES')
    cols = ['species_id'] + list(TAXON_COLS)
    df = pd.read_excel(f)[cols].set_index('species_id')
    return {v: df[c].to_dict() for c, v in TAXON_COLS.items()}

In [ ]:
#| export
def add_taxon_info(
    dfs:dict       # dict of DataFrames keyed by sample group
):
    "Add taxon columns to BIOTA from species lookup."
    if 'BIOTA' not in dfs: return
    cols = get_taxon_cols()
    for col, lut in cols.items():
        dfs['BIOTA'][col] = dfs['BIOTA']['SPECIES'].map(lut).fillna('Unknown')

In [ ]:
test = {'BIOTA': pd.DataFrame({'SPECIES': [99, 96]})}
add_taxon_info(test)
test_eq(test['BIOTA']['TAXONNAME'].tolist(), ['Gadus morhua', 'Fucus vesiculosus'])
test_eq(test['BIOTA']['TAXONRANK'].tolist(), ['species', 'species'])

### Map lookup-table columns

Convert integer enum IDs to display names. DL and FILT use the Excel `name` column rather than the sanitised version.

In [ ]:
#| export
def map_lut(
    dfs:dict,             # dict of DataFrames keyed by sample group
    cols:list,            # Column names to map
    key:str='name',       # LUT key column
    value:str='id',       # LUT value column
    reverse:bool=True     # Reverse the mapping direction
):
    "Map columns using get_lut."
    for col in cols:
        lut = get_lut(col, key=key, value=value, reverse=reverse)
        for df in dfs.values():
            if col in df.columns: df[col] = df[col].map(lut)

In [ ]:
test = {'SEAWATER': pd.DataFrame({'DL': [1, 2, 3]})}
map_lut(test, ['DL'])
test_eq(test['SEAWATER']['DL'].tolist(), ['=', '<', 'ND'])

In [ ]:
test = {'SEAWATER': pd.DataFrame({'FILT': [0, 1, 2]})}
map_lut(test, ['FILT'])
test_eq(test['SEAWATER']['FILT'].tolist(), ['Not available', 'Yes', 'No'])

In [ ]:
test = {'SEAWATER': pd.DataFrame({'VALUE': [1]})}
map_lut(test, ['DL'])
test_eq(list(test['SEAWATER'].columns), ['VALUE'])

### Decode remaining enumerated columns

`CSV_DTYPES` marks decoded vs encoded columns. DL and FILT are excluded since they use a different LUT column.

In [ ]:
#| export
def decode_csv_vars(
    dfs:dict       # dict of DataFrames keyed by sample group
):
    "Decode enumerated columns marked as `state='decoded'` in CSV_DTYPES."
    decoded = [c for c,cfg in CSV_DTYPES.items() if cfg['state']=='decoded' and c not in ('DL','FILT')]
    for col in decoded:
        lut = get_lut(col, reverse=True)
        for df in dfs.values():
            if col in df.columns: df[col] = df[col].map(lut)

::: {.callout-note}

#### Note for MARIS DB team
It would be cleaner/more consistent to expect all MARIS LUT values in their encoded (integer) form during SQL import, rather than receiving a mix of decoded display names and encoded IDs.
:::

### Write CSV files

Rename columns via `CSV_VARS`, then write one file per group.

In [ ]:
#| export
def to_csv_files(
    dfs:dict,              # dict of DataFrames keyed by sample group
    fname_in:str,          # Input NetCDF file path
    dest_out:str=None      # Destination path stem; defaults to fname_in stem
):
    "Rename columns and write one CSV per group."
    fstem = Path(dest_out or Path(fname_in).with_suffix(''))
    paths = []
    for grp, df in dfs.items():
        df = df.rename(columns=CSV_VARS)
        p = Path(f'{fstem}_{grp}.csv')
        df.to_csv(p, index=False)
        paths.append(p)
    return paths

In [ ]:
test = {'SEAWATER': pd.DataFrame({'TIME': [1], 'VALUE': [2]})}
fname = Path('/tmp/test_nc2csv')
to_csv_files(test, fname)
res = pd.read_csv(f'{fname}_SEAWATER.csv')
test_eq(list(res.columns), ['begperiod', 'activity'])
Path(f'{fname}_SEAWATER.csv').unlink()

## Entry point

`to_csv` composes the helpers in order.

In [ ]:
#| export
def to_csv(
    fname_in:str,          # MARIS standard NetCDF file path
    dest_out:str=None,     # Destination path stem; defaults to parent of fname_in
    ref_id:int=None        # Reference ID to add as REF_ID column
):
    "Convert MARIS standard NetCDF file to import-ready CSV files."
    decode_time(read_nc_groups(fname_in))
    add_sample_type(dfs)
    add_ref_id(dfs, ref_id)
    add_taxon_info(dfs)
    map_lut(dfs, ['DL', 'FILT'])
    decode_csv_vars(dfs)
    return to_csv_files(dfs, fname_in, dest_out)

### Usage example

Convert a MARIS standard NetCDF file to import-ready CSV files using `to_csv`.

In [ ]:
#| eval: false
fname_in = Path('files/nc/100_HELCOM_MORS_2024.nc')
to_csv(fname_in, ref_id=100)

[Path('files/nc/100_HELCOM_MORS_2024_BIOTA.csv'),
 Path('files/nc/100_HELCOM_MORS_2024_SEAWATER.csv'),
 Path('files/nc/100_HELCOM_MORS_2024_SEDIMENT.csv')]

`to_csv` produces one CSV per sample type group found in the NetCDF file.

In [ ]:
#| eval: false
for f in sorted(Path('files/nc/').glob('100_HELCOM_MORS_2024_*.csv')):
    print(f.name)

100_HELCOM_MORS_2024_BIOTA.csv


100_HELCOM_MORS_2024_SEAWATER.csv


100_HELCOM_MORS_2024_SEDIMENT.csv


The output CSV includes 23 columns: sample metadata, activity, detection limit info, taxon details for biota, and reference ID. Below the first row:

In [ ]:
#| eval: false
df = pd.read_csv('files/nc/100_HELCOM_MORS_2024_BIOTA.csv')
print(df.head(1).T)

                                                 0
samplabcode                           BBFFG1999001
longitude                                    13.72
latitude                                     54.22
sampdepth                                      0.0
begperiod                                929404800
station                                     BGBODD
nuclide_id                                       4
activity                                     841.0
unit_id                                          4
uncertaint                                   58.87
detection                                        =
BIO_GROUP                                       11
species_id                                      96
bodypar_id                                      54
drywt                                          NaN
wetwt                                          NaN
percentwt                                   0.1692
samptype_id                                      2
ref_id                         